In [1]:
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import example_loader as el
import numpy as np
import scipy.sparse as sps
import scipy.sparse.linalg as spsl

In [23]:
# A function to create cuts given a target point
def add_balas_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx):
    # for each column in the tableau
    # construct a sparse vector for it
    # get the length of that vector via norm1 (plus 1 if we're an int column)
    # add our cut: sum_j(x_j/a_j)
    
    norm = 1
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    print("   Gap to target:", radius, ":", current[:7], "to", target[:7])
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var = gu.read_tableau(relaxed, basis)
    
    # norm 1 lengths:
    lengths = np.linalg.norm(tableau, norm, axis=0)
    lengths[list(integer_idx.keys())] += 1  # integer_idx is {original var idx : idx in int_var list} 
    lengths /= radius
        
    variables = relaxed.getVars()
    inequalities = [con for con in relaxed.getConstrs() if con.Sense != '=']  # not sure if I should filter out =
    def find_variable(index):
        if index < len(variables):
            return variables[index]
        # if only gurobi gave us access to their slack variables....
        # instead, we have to solve for it:
        constraint = inequalities[index - len(variables)]
        lhs, sense, rhs = relaxed.getRow(constraint), constraint.Sense, constraint.RHS
        assert sense == '>'
        return lhs - rhs
    
    
    relaxed.addConstr(gp.quicksum(lengths[i] * find_variable(j) for i, j in enumerate(col_to_var)) >= 1)
    return True


In [24]:
# a function to run cuts against the nearest integer:
def run_cuts_to_nearest_int(instances, cut_function, loops=5):
    for instance in instances:
        m: gp.Model = instance.as_gurobi_model()
        m.params.LogToConsole = 0
        gu.standardize_lt_to_gt(m)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(m)
        print("Running model:", m.ModelName)
        for i in range(loops):
            m.optimize()
            nearest = gu.nearest_integer(int_vars)
            if not cut_function(m, nearest, int_vars, int_idx):
                print("Final score of relaxed:", m.getObjective().getValue())
                break


In [25]:
# test the cuts on simple examples:
run_cuts_to_nearest_int(el.get_instances().values(), add_balas_ball_cut)

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Negated 2 constraints on 2D from bottom
Relaxed 2 variables on 2D from bottom
Running model: 2D from bottom
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.2888888888888886 : [ 2.91111111 -0.2       ] to [ 3. -0.]


AssertionError: We can only pull this data from models solved to optimum.

In [7]:
# a function to find the hyperplane closest to a point
def compute_hyperplane_via_lp(x0, b0, tableau):
    m = gp.Model()
    b = m.addVar(lb=-gp.GRB.INFINITY)
    w = m.addMVar(x0.shape, lb=-gp.GRB.INFINITY)
    wn = m.addVar()
    m.addConstr(wn == gp.norm(w, 1))
    m.addConstr(wn == 1)  # optional
    z = m.addVar()
    m.addConstr(z >= x0 @ w - b - b0)
    m.addConstr(z >= -x0 @ w + b + b0)
    m.setObjective(z)
    m.addConstrs(vec @ w <= b for vec in tableau.T)
    m.params.LogToConsole = 0
    m.optimize()
    return w.X, b.X

# create a function to do cuts via LP:
def add_lp_ball_cut(relaxed: gp.Model, target, tableau, basis, integer_vars, integer_idx):
    rx = integer_vars.X
    ex = target
    tol = relaxed.params.FeasibleTol
    
    # if it matches the other, then bail out:
    if np.all(np.isclose(rx, ex, 0.0, tol)):
        return False
    dist = np.linalg.norm(ex - rx, 1)
    print("   Gap to target:", dist, ":", rx[:7], "to", ex[:7])

    # read the corner of the polytope
    tableau, col_to_var = gu.read_tableau(relaxed, basis)  # TODO: only read the tied-to-integer rows, not all rows

    # expand tableau to include the other variables: TODO: pick the right sparse format, construct it before read_tab
    tableau2 = sps.lil_matrix((len(integer_idx), tableau.shape[1]))
    for i, base in enumerate(basis):
        if base in integer_idx:
            tableau2[integer_idx[base]] = tableau[i]
    for c, v in enumerate(col_to_var):  # TODO: get rid of these Python loops
        if v in integer_idx:
            tableau2[integer_idx[v], c] = -1
    # delete tableau here?

    # normalize the columns:
    tableau2 = tableau2 / spsl.norm(tableau2, 1, axis=0) * dist

    # generate the LP to find the hyperplane
    w, b = compute_hyperplane_via_lp(ex-rx, dist, tableau2)

    # add the cut:
    relaxed.addConstr(w @ (integer_vars - rx) >= b)
    return True
